In [11]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))

import config
import pandas as pd
df = pd.read_csv(config.DB_LOCATION)

C:\Users\bnpar\AppData\Local\Temp\ipykernel_33320\2436533794.py:10: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(config.DB_LOCATION)


In [12]:
#reduced dataframe
red_df = df[['Name', 'Sex', 'Division', 'BodyweightKg', 'WeightClassKg','Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 'TotalKg', 'Date', 'Sanctioned', 'Equipment', 'Event']]

In [13]:
def get_mixed_cols(dataframe):
    all_types_count = {} #nested dict w column heading as outer key. Inner key-value pairs are data types/how many entries with that datatyoe
    for col in dataframe.columns:
        col_types = dataframe[col].map(type)   #applies type(x) to every x in the column so you get series ['string', 'string', 'float'....]
        all_types_count[col] = col_types.value_counts().to_dict()
    mixed_type_cols = []
    for col in all_types_count:
        no_types = len(all_types_count[col].values())
        #we are only appending columns with mixed data types
        if no_types > 1:
            mixed_type_cols.append(col)
    return mixed_type_cols

In [14]:
mixed_cols = get_mixed_cols(red_df)
print(mixed_cols)

['Division', 'WeightClassKg']


In [15]:
#checking if all the floats are because of NaN
for col in mixed_cols:
    mask = red_df[col].map(type) == float
    float_col = red_df.loc[mask, col]
    float_col = float_col.dropna()
    print(float_col)

Series([], Name: Division, dtype: object)
Series([], Name: WeightClassKg, dtype: object)


- All float datatypes are because of NaN entries in WeightClassKg or Division. Want to drop any entries where we dont know what the weight class is.
- Are okay with not knowing the Division i.e. whether the lifter is a Junior or Open lifter for now, as a lifter of any age can set an Open WR.
- Chosen to drop any unsanctioned meets.
- Flitered for Raw powerlifting only (specified in 'Equipment' column).
- Turned dates into datetime objects.
- Filtering for full power events (e.g. excluding bench only comps)

In [16]:
red_df = red_df[red_df['Equipment'] == 'Raw'].copy()
red_df['Date'] = pd.to_datetime(red_df['Date'] , format = '%Y-%m-%d')
red_df = red_df.dropna(axis = 'rows', subset = ['WeightClassKg'])
unsanctioned = red_df[red_df['Sanctioned'] == 'No'].index.to_list()
red_df = red_df.drop(labels = unsanctioned, axis = 'index')
full_power = red_df[red_df['Event'] == 'SBD'].copy()
#bench_only = red_df[red_df['Event']== 'B'].copy()

Next we look for weight classes in the df which are not the weight classes that the IPF and its affiliates currrently use. Created a dictionary of current world record totals so that I can flag totals in dataframe which are more than the official world records. Note that the total being more than the official WR does not necessarily mean the entry is incorrect. Official WRs can only be set at international meets and lifters may have exceeded this at national or local events, without setting a new official WR.

Also note that the 53kg and 43kg in the men's and women's categories respectively are only in the Junior age division.


In [ ]:
m_off_wr_totals = {'53': 559, '59': 669.5, '66': 770, '74': 843, '83': 890, '93': 918, '105': 975.5, '120': 978.5, '120+': 1152.5}
w_off_wr_totals = {'43': 349.5, '47': 435, '52': 482.5, '57': 520, '63': 565, '69': 628, '76': 620, '84': 647, '84+': 737.5}

exceeds_wr = pd.DataFrame(columns = red_df.columns.to_list())

for weight_class in m_off_wr_totals:
    weight_class_entries = full_power[(full_power['WeightClassKg'] == weight_class) & (full_power['Sex'] == 'M')].copy()
    exceeds_class_wr = weight_class_entries[weight_class_entries['TotalKg']>m_off_wr_totals[weight_class] ]
    exceeds_wr = pd.concat([exceeds_wr, exceeds_class_wr])

for weight_class in w_off_wr_totals:
    weight_class_entries = full_power[(full_power['WeightClassKg'] == weight_class) & (full_power['Sex'] == 'F')].copy()
    exceeds_class_wr = weight_class_entries[weight_class_entries['TotalKg']>w_off_wr_totals[weight_class]]
    exceeds_wr = pd.concat([exceeds_wr, exceeds_class_wr])
    
exceeds_wr

All entries that exceed the current world record are credible and correct against the results from those meets.

In [24]:
invalid_class_entries = full_power[full_power['WeightClassKg'].apply(lambda x: (x not in m_off_wr_totals.keys()) and (x not in w_off_wr_totals.keys()) )]
invalid_class = invalid_class_entries['WeightClassKg'].unique()
#inserting year column
invalid_class_entries.insert(len(invalid_class_entries.columns.to_list()), 'Year', invalid_class_entries['Date'].dt.year)

f'{(len(invalid_class_entries)/len(full_power))*100 :.2f}% of the dataset is entries that do not match the IPF\'s current weight classes'

"7.01% of the dataset is entries that do not match the IPF's current weight classes"

Next we want the number of entries in each invalid class to determine whether they are mistakes or a result of the IPF changing the weight classes.

In [ ]:
num_invalid_class_entries = invalid_class_entries.groupby(['WeightClassKg']).size()
print(len(full_power))


In [ ]:
# want number of lifters competing in 'invalid' weight classes every year 
no_lifters_per_year = pd.DataFrame(columns = ['WeightClassKg', 'Year', 'NumberOfLifters'])

In [53]:
#getting unique combinations of name, class and year
# unique_name_class_year = invalid_class_entries[['Name', 'WeightClassKg', 'Year', 'Sex']].drop_duplicates()
unique_name_class_year = invalid_class_entries[['Name', 'WeightClassKg', 'Year', 'Sex']]

invalid_class_counts = unique_name_class_year.groupby(['WeightClassKg', 'Year']).size().reset_index(name = 'NumberOfLifters').sort_values('Year', ignore_index = True)
invalid_class_counts[invalid_class_counts['WeightClassKg']=='56']


,WeightClassKg,Year,NumberOfLifters
0,56,1966,1
1,56,1967,1
2,56,1969,1
3,56,1970,1
4,56,1971,1
5,56,1972,1
8,56,1973,3
13,56,1974,3
21,56,1975,1
22,56,1976,1


In [ ]:
# classes_per_year = pd.DataFrame(columns = ['Year', 'Classes'])

# dictonary = {} #class as key: years as values
# for w_class in invalid_class:
#     class_df = full_power[full_power['WeightClassKg'] == w_class]
#     class_df['Year'] = class_df['Date'].dt.year
#     for year in class_df['Year'].unique():
#         print(f'{len(class_df[class_df['Year']== year])} competitions were done by people in the {w_class}kg weight class in {year}')

In [ ]:
mask = (red_df['TotalKg'].between(0,1153)).apply(lambda x: not x)
print(f'length is {len(red_df)}')
outliers = red_df[mask]
print(f'no outliers is {len(outliers)}')
no_bomb_out = outliers.dropna()
no_bomb_out.head(10)

In [ ]:
# copy = red_df.copy()
# copy['Year'] =  copy['Date'].dt.year
# copy['Year'].value_counts()